# Pandalyzer — Manual Cleaning Tests
## Purpose
Exercise the existing cleaner API using synthetic, controlled examples and the messy retail CSV when available. Run cells from top to bottom. The original CSV is never overwritten.

## 1. Imports and dataset setup
Load Pandalyzer functions and the dataset. This cell also works in the provided test environment.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "database").is_dir() and (p / "src" / "autoclean").is_dir()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from autoclean.cleaner import remove_duplicate_rows, handle_missing_values, clean_dataframe
from autoclean.profiler import profile_dataframe, compare_data_quality
from autoclean.validators import validate_missing_values, validate_required_columns, validate_column_types, validate_numeric_ranges
from autoclean.connectors import load_csv, save_csv, save_quality_report

csv_path = PROJECT_ROOT / "database" / "messy_retail_orders.csv"
print("Project root:", PROJECT_ROOT)
print("Dataset available:", csv_path.exists())

if csv_path.exists():
    df = load_csv(csv_path)
    print("Dataset shape:", df.shape)
    print("Missing cells:", int(df.isna().sum().sum()))
    print("Exact duplicate rows:", int(df.duplicated().sum()))
    display(df.head())
else:
    df = pd.DataFrame({"order_id":[1,2,3], "amount":[10.0,None,30.0]})
    print("CSV unavailable; using a small fallback DataFrame for API tests.")

Project root: /run/media/amateur/MATRIX/Pandalyzer/AutoClean
Dataset available: True
Dataset shape: (10000, 20)
Missing cells: 23018
Exact duplicate rows: 0


,order_id,order_date,customer_id,product_id,category,quantity,unit_price,discount_pct,order_amount,payment_method,order_status,city,state,sales_channel,customer_email,delivery_days,rating,coupon_code,is_returned,notes
0,ORD009826,2024-06-19 16:56:11,CUST01648,NaN,Grocery,5.0,96.55,30.0,NaN,Credit Card,Cancelled,Mumbai,Maharashtra,Website,not-an-email,12.0,1.0,NaN,No,NaN
1,ORD004017,2025-03-18 10:42:56,CUST01249,\tPRD0244\t,Home & Kitchen,NaN,1567.44,5.0,4467.2,COD,\tShipped\t,Hyderabad,Telangana,Website,customer1175@example.com,NaN,3.0,NaN,No,NaN
2,ORD008732,2026-09-30 12:23:18,CUST00211,PRD0160,Clothing,2.0,958.08,0.0,1916.16,Credit Card,Returned,lucknow,Uttar Pradesh,Store,NaN,11.0,1.0,NaN,No,Gift order
3,ORD009698,2024-01-18 19:02:44,CUST00406,PRD0178,Electronics,2.0,421.03,0.0,NaN,Wallet,Returned,Bengaluru,NaN,NaN,@example.com,9.0,2.0,FESTIVE25,NaN,Customer requested update
4,ORD004226,NaN,CUST00610,PRD0235,Sports,2.0,1653.18,20.0,2645.09,Cash,Cancelled,Delhi,Delhi,Website,\tcustomer780@example.com \t,8.0,5.0,NaN,NaN,Gift order


## 2. Test duplicate removal
Verify duplicate rows are removed, first occurrence is kept, and the input DataFrame is unchanged.

In [2]:
duplicate_test_df = pd.DataFrame({"order_id":[101,102,102,103], "quantity":[2,1,1,4], "amount":[500,250,250,800]}, index=[5,6,7,8])
before = duplicate_test_df.copy(deep=True)
deduplicated_df = remove_duplicate_rows(duplicate_test_df)
display(deduplicated_df)
assert len(deduplicated_df) == 3
assert deduplicated_df.index.tolist() == [5,6,8]
pd.testing.assert_frame_equal(duplicate_test_df, before)
print("PASS: duplicate removal and non-mutation")

,order_id,quantity,amount
5,101,2,500
6,102,1,250
8,103,4,800


PASS: duplicate removal and non-mutation


## 3. Test drop_rows strategy
Rows with any missing cells should be removed.

In [3]:
missing_test_df = pd.DataFrame({"quantity":[2,None,4,6], "amount":[100,200,None,300]})
drop_rows_result = handle_missing_values(missing_test_df, strategy="drop_rows")
display(drop_rows_result)
assert len(drop_rows_result) == 2 and drop_rows_result.isna().sum().sum() == 0
pd.testing.assert_frame_equal(missing_test_df, pd.DataFrame({"quantity":[2,None,4,6], "amount":[100,200,None,300]}))
print("PASS: drop_rows")

,quantity,amount
0,2.0,100.0
3,6.0,300.0


PASS: drop_rows


## 4. Test fill_constant strategy
All missing cells should be filled with the specified constant.

In [4]:
constant_result = handle_missing_values(missing_test_df, strategy="fill_constant", value=0)
display(constant_result)
assert constant_result.isna().sum().sum() == 0
assert constant_result.loc[1,"quantity"] == 0 and constant_result.loc[2,"amount"] == 0
print("PASS: fill_constant")

,quantity,amount
0,2.0,100.0
1,0.0,200.0
2,4.0,0.0
3,6.0,300.0


PASS: fill_constant


## 5. Test fill_mean strategy
Missing numeric values should be replaced by their column means.

In [5]:
mean_result = handle_missing_values(missing_test_df, strategy="fill_mean")
display(mean_result)
assert mean_result.loc[1,"quantity"] == 4
assert mean_result.loc[2,"amount"] == 200
assert mean_result.isna().sum().sum() == 0
print("PASS: fill_mean")

,quantity,amount
0,2.0,100.0
1,4.0,200.0
2,4.0,200.0
3,6.0,300.0


PASS: fill_mean


## 6. Test fill_median strategy
Missing numeric values should be replaced by column medians.

In [6]:
median_result = handle_missing_values(missing_test_df, strategy="fill_median")
display(median_result)
assert median_result.loc[1,"quantity"] == 4
assert median_result.loc[2,"amount"] == 200
assert median_result.isna().sum().sum() == 0
print("PASS: fill_median")

,quantity,amount
0,2.0,100.0
1,4.0,200.0
2,4.0,200.0
3,6.0,300.0


PASS: fill_median


## 7. Test the combined cleaning pipeline
Check that duplicate removal happens before missing-value handling and that the input is preserved.

In [7]:
pipeline_input = pd.DataFrame({"id":[1,2,2,3], "value":[10.0,20.0,20.0,None]})
pipeline_before = pipeline_input.copy(deep=True)
pipeline_result = clean_dataframe(pipeline_input, remove_duplicates=True, missing_strategy="drop_rows")
display(pipeline_result)
assert len(pipeline_result) == 2
assert pipeline_result.isna().sum().sum() == 0
assert pipeline_result.duplicated().sum() == 0
pd.testing.assert_frame_equal(pipeline_input, pipeline_before)
print("PASS: combined cleaning pipeline")

,id,value
0,1,10.0
1,2,20.0


PASS: combined cleaning pipeline


## 8. Run a conservative real-dataset trial
This trial drops incomplete rows, which may discard useful records. It is for testing only; it does not overwrite the source.

In [8]:
real_df = load_csv(csv_path) if csv_path.exists() else df.copy()
real_before = real_df.copy(deep=True)
real_cleaned = clean_dataframe(real_df, remove_duplicates=True, missing_strategy="drop_rows")
print("Before:", real_df.shape, "| missing:", int(real_df.isna().sum().sum()), "| duplicates:", int(real_df.duplicated().sum()))
print("After: ", real_cleaned.shape, "| missing:", int(real_cleaned.isna().sum().sum()), "| duplicates:", int(real_cleaned.duplicated().sum()))
pd.testing.assert_frame_equal(real_df, real_before)
assert real_cleaned.isna().sum().sum() == 0
assert real_cleaned.duplicated().sum() == 0
print("PASS: real-data trial")

Before: (10000, 20) | missing: 23018 | duplicates: 0
After:  (688, 20) | missing: 0 | duplicates: 0
PASS: real-data trial
